# 01c — Mỗi thành viên crawl một phần riêng

**CPU.** Mỗi người dùng Drive riêng được, nhưng phải có cùng snapshot `links_corpus.parquet` và cùng commit code. Chỉ sửa `TEAM_MEMBER` ở ô dưới thành 1, 2 hoặc 3. Khoảng dòng được tính từ Parquet và khớp ranh giới shard 512; không lấy `doc_id` làm số thứ tự.

Chưa chạy full nếu cổng quy mô chưa duyệt. Sau khi duyệt, chạy lại cùng `TEAM_MEMBER`, `RUN_PREFIX`, commit và cấu hình để tiếp tục checkpoint.


Notebook tùy chọn, không thuộc luồng chính 00–04. Bootstrap dùng optional_data_code_lock.json riêng. Runtime mới chạy lại từ đầu để resume. Khi chạy team, cả ba ghim cùng full SHA vào CODE_REVISION trước khi crawl. Build bằng code đã merge dùng tên run mới; notebook 03 phải dùng cùng full CODE_COMMIT khi freeze.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "optional_data_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
reference = CODE_REVISION or lock.get("git_commit") or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "optional_data_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Chọn người và xem phần được giao

In [ ]:
import sys
sys.path.insert(0, str(CHECKOUT / "scripts"))
from team_crawl import team_plan
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.crawl import CrawlConfig

TEAM_MEMBER = 1  # Người 1 đặt 1, người 2 đặt 2, người 3 đặt 3
RUN_PREFIX = "full-team-v1"  # Cả ba phải giống hệt nhau; đổi tên cho đợt mới
SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
CONFIG = CrawlConfig(**PIPELINE_CONFIG["crawler"])
PLAN = team_plan(OFFICIAL_LINKS, workers=3, shard_size=CONFIG.shard_size, run_prefix=RUN_PREFIX)
if TEAM_MEMBER not in (1, 2, 3):
    raise ValueError("TEAM_MEMBER phải là 1, 2 hoặc 3")
PART = PLAN["parts"][TEAM_MEMBER - 1]
print("Plan hash:", PLAN["plan_sha256"])
print("Official rows:", PLAN["total_rows"], "| official SHA-256:", PLAN["links_sha256"])
print("Phần của bạn:", PART)
print("Code commit:", CODE_COMMIT)


## 2. Kiểm cổng quy mô, rồi crawl phần riêng

Ô này thực sự truy cập URL. Chỉ chạy sau khi 3 người đã đối chiếu cùng Plan hash, official SHA-256, commit và mốc duyệt của repo đã hoàn tất. Mỗi người chỉ ghi vào run riêng của mình. Nếu Colab ngắt, chạy lại cùng thiết lập để tiếp tục.


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json, code_fingerprint, sha256_file
from vietmedbridge.crawl import crawl_links
from vietmedbridge.gates import authorize_scale
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.golden import run_golden_suite

golden_path = DATA_ROOT / "reports/golden_latest.json"
golden = read_json(golden_path) if golden_path.exists() else {}
fixture = CHECKOUT / "tests/golden/cases.json"
if (not golden.get("passed") or golden.get("code_sha256") != code_fingerprint()
        or golden.get("fixture_sha256") != sha256_file(fixture)
        or golden.get("chunking") != PIPELINE_CONFIG["chunking"]):
    tokenizer_lock = read_json(DATA_ROOT / "tokenizer_lock.json")
    tokenizer, tokenizer_spec = load_bge_tokenizer(tokenizer_lock["revision"])
    golden = run_golden_suite(fixture, tokenizer, tokenizer_spec,
                              ChunkConfig(**PIPELINE_CONFIG["chunking"]))
    atomic_json(golden_path, golden)

# Kiểm theo quy mô TOÀN dự án, không chỉ ~1/3 số dòng của cá nhân.
authorize_scale(DATA_ROOT, rows=PLAN["total_rows"],
                corpus_sha256=SNAPSHOT["files"]["links_corpus.parquet"]["sha256"],
                golden_report=golden, chunking=PIPELINE_CONFIG["chunking"])
SUMMARY = await crawl_links(
    OFFICIAL_LINKS, DATA_ROOT / "crawl", run_name=PART["run_name"],
    config=CONFIG, start=PART["start_row"], stop=PART["stop_row"],
    max_shards=None, work_dir=WORK_DIR,
    origin_corpus_sha256=SNAPSHOT["files"]["links_corpus.parquet"]["sha256"],
)
print(json.dumps({key: SUMMARY[key] for key in (
    "run_name", "recorded_documents", "requested_input_records", "range_complete", "statuses"
)}, ensure_ascii=False, indent=2))
print("Thư mục bàn giao:", DATA_ROOT / "crawl" / PART["run_name"])


## Bàn giao

Mỗi người gửi cho người ghép **toàn bộ thư mục run riêng** (`run.json` và `parts/`), cùng `Plan hash` và `Code commit`. Không chỉ gửi `summary.json`. Trước khi ghép, cả ba run phải báo `range_complete: true`. Có thể dùng Shared Drive hoặc chuyển thư mục; bản gốc của mỗi người được giữ nguyên.
